In [1]:
from pyspark.sql import functions as F
from pyspark.sql.functions import (
    col,
    upper,
    trim,
    lit,
    lpad,
    concat_ws,
    to_date
)
from pyspark.sql.window import Window

StatementMeta(, 66d20bab-0403-4f11-afbd-0bfce45d5dc3, 3, Finished, Available, Finished, False)

In [2]:
fuel = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("Files/raw/batch/form41/fuel_cost_consumption_2024_jan.csv")
)

dim_airline = spark.table("dim_airline")
dim_date = spark.table("dim_date")

print("Fuel Source Rows:", fuel.count())
print("Airline Dimension Rows:", dim_airline.count())
print("Date Dimension Rows:", dim_date.count())

fuel.printSchema()

StatementMeta(, 66d20bab-0403-4f11-afbd-0bfce45d5dc3, 4, Finished, Available, Finished, False)

Fuel Source Rows: 53
Airline Dimension Rows: 155
Date Dimension Rows: 31
root
 |-- YEAR: integer (nullable = true)
 |-- MONTH: integer (nullable = true)
 |-- AIRLINE_ID: integer (nullable = true)
 |-- UNIQUE_CARRIER: string (nullable = true)
 |-- CARRIER_NAME: string (nullable = true)
 |-- TDOMT_GALLONS: double (nullable = true)
 |-- TINT_GALLONS: double (nullable = true)
 |-- TOTAL_GALLONS: double (nullable = true)
 |-- TDOMT_COST: double (nullable = true)
 |-- TINT_COST: double (nullable = true)
 |-- TOTAL_COST: double (nullable = true)



In [3]:
fuel_std = (
    fuel
    .select(
        col("YEAR").cast("int").alias("year"),
        col("MONTH").cast("int").alias("month"),

        col("AIRLINE_ID").cast("long").alias("airline_id"),
        upper(trim(col("UNIQUE_CARRIER"))).alias("unique_carrier"),
        trim(col("CARRIER_NAME")).alias("carrier_name"),

        col("TDOMT_GALLONS").cast("double").alias("domestic_gallons"),
        col("TINT_GALLONS").cast("double").alias("international_gallons"),
        col("TOTAL_GALLONS").cast("double").alias("total_gallons"),

        col("TDOMT_COST").cast("double").alias("domestic_cost"),
        col("TINT_COST").cast("double").alias("international_cost"),
        col("TOTAL_COST").cast("double").alias("total_cost")
    )
    .withColumn(
        "operation_date",
        to_date(
            concat_ws(
                "-",
                col("year"),
                lpad(col("month"), 2, "0"),
                lit("01")
            )
        )
    )
)

print("Standardized Fuel Rows:", fuel_std.count())
fuel_std.printSchema()

StatementMeta(, 66d20bab-0403-4f11-afbd-0bfce45d5dc3, 5, Finished, Available, Finished, False)

Standardized Fuel Rows: 53
root
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- airline_id: long (nullable = true)
 |-- unique_carrier: string (nullable = true)
 |-- carrier_name: string (nullable = true)
 |-- domestic_gallons: double (nullable = true)
 |-- international_gallons: double (nullable = true)
 |-- total_gallons: double (nullable = true)
 |-- domestic_cost: double (nullable = true)
 |-- international_cost: double (nullable = true)
 |-- total_cost: double (nullable = true)
 |-- operation_date: date (nullable = true)



In [4]:
airline_lookup = (
    dim_airline
    .select(
        "airline_key",
        "airline_id"
    )
)

date_lookup = (
    dim_date
    .select(
        "date_key",
        col("flight_date").alias("operation_date")
    )
)

print("Lookups created.")

StatementMeta(, 66d20bab-0403-4f11-afbd-0bfce45d5dc3, 6, Finished, Available, Finished, False)

Lookups created.


In [5]:
fuel_enriched = (
    fuel_std.alias("f")

    .join(
        airline_lookup.alias("a"),
        col("f.airline_id") == col("a.airline_id"),
        "left"
    )

    .join(
        date_lookup.alias("d"),
        col("f.operation_date") == col("d.operation_date"),
        "left"
    )

    .select(
        col("d.date_key"),
        col("a.airline_key"),

        col("f.airline_id"),
        col("f.unique_carrier"),
        col("f.carrier_name"),

        col("f.total_gallons"),
        col("f.domestic_gallons"),
        col("f.international_gallons"),

        col("f.total_cost"),
        col("f.domestic_cost"),
        col("f.international_cost")
    )
)

StatementMeta(, 66d20bab-0403-4f11-afbd-0bfce45d5dc3, 7, Finished, Available, Finished, False)

In [6]:
source_rows = fuel_std.count()
joined_rows = fuel_enriched.count()

print("Source Rows:", source_rows)
print("Rows After Dimension Joins:", joined_rows)
print("Row Difference:", joined_rows - source_rows)

StatementMeta(, 66d20bab-0403-4f11-afbd-0bfce45d5dc3, 8, Finished, Available, Finished, False)

Source Rows: 53
Rows After Dimension Joins: 53
Row Difference: 0


In [7]:
fuel_key_validation = fuel_enriched.agg(
    F.sum(
        F.when(col("date_key").isNull(), 1).otherwise(0)
    ).alias("null_date_keys"),

    F.sum(
        F.when(col("airline_key").isNull(), 1).otherwise(0)
    ).alias("null_airline_keys")
)

fuel_key_validation.show(truncate=False)

StatementMeta(, 66d20bab-0403-4f11-afbd-0bfce45d5dc3, 9, Finished, Available, Finished, False)

+--------------+-----------------+
|null_date_keys|null_airline_keys|
+--------------+-----------------+
|0             |0                |
+--------------+-----------------+



In [8]:
print(
    "Distinct Source Airline IDs:",
    fuel_std
    .select("airline_id")
    .distinct()
    .count()
)

print(
    "Distinct Conformed Airline Keys:",
    fuel_enriched
    .select("airline_key")
    .distinct()
    .count()
)

print(
    "Duplicate Airline-Month Grain Rows:",
    fuel_enriched
    .groupBy(
        "date_key",
        "airline_key"
    )
    .count()
    .filter(col("count") > 1)
    .count()
)

StatementMeta(, 66d20bab-0403-4f11-afbd-0bfce45d5dc3, 10, Finished, Available, Finished, False)

Distinct Source Airline IDs: 53
Distinct Conformed Airline Keys: 53
Duplicate Airline-Month Grain Rows: 0


In [9]:
missing_airlines = (
    fuel_enriched
    .filter(col("airline_key").isNull())
    .select(
        "airline_id",
        "unique_carrier",
        "carrier_name"
    )
    .distinct()
    .orderBy("airline_id")
)

missing_airlines.show(truncate=False)

StatementMeta(, 66d20bab-0403-4f11-afbd-0bfce45d5dc3, 11, Finished, Available, Finished, False)

+----------+--------------+------------+
|airline_id|unique_carrier|carrier_name|
+----------+--------------+------------+
+----------+--------------+------------+



In [10]:
fuel_key_window = Window.orderBy(
    "date_key",
    "airline_key"
)

fact_fuel_consumption_new = (
    fuel_enriched
    .withColumn(
        "fuel_key",
        F.row_number().over(fuel_key_window)
    )
    .select(
        "fuel_key",
        "date_key",
        "airline_key",

        "total_gallons",
        "domestic_gallons",
        "international_gallons",

        "total_cost",
        "domestic_cost",
        "international_cost"
    )
)

StatementMeta(, 66d20bab-0403-4f11-afbd-0bfce45d5dc3, 12, Finished, Available, Finished, False)

In [11]:
print("Final Fuel Rows:", fact_fuel_consumption_new.count())

print(
    "Distinct Fuel Keys:",
    fact_fuel_consumption_new
    .select("fuel_key")
    .distinct()
    .count()
)

print(
    "Null Airline Keys:",
    fact_fuel_consumption_new
    .filter(col("airline_key").isNull())
    .count()
)

print(
    "Distinct Airline Keys:",
    fact_fuel_consumption_new
    .select("airline_key")
    .distinct()
    .count()
)

fact_fuel_consumption_new.printSchema()

StatementMeta(, 66d20bab-0403-4f11-afbd-0bfce45d5dc3, 13, Finished, Available, Finished, False)

Final Fuel Rows: 53
Distinct Fuel Keys: 53
Null Airline Keys: 0
Distinct Airline Keys: 53
root
 |-- fuel_key: integer (nullable = false)
 |-- date_key: integer (nullable = true)
 |-- airline_key: integer (nullable = true)
 |-- total_gallons: double (nullable = true)
 |-- domestic_gallons: double (nullable = true)
 |-- international_gallons: double (nullable = true)
 |-- total_cost: double (nullable = true)
 |-- domestic_cost: double (nullable = true)
 |-- international_cost: double (nullable = true)



In [12]:
(
    fact_fuel_consumption_new
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("fact_fuel_consumption")
)

print("fact_fuel_consumption persisted successfully.")

persisted_fuel = spark.table("fact_fuel_consumption")

print("Persisted Fuel Rows:", persisted_fuel.count())

print(
    "Persisted Distinct Fuel Keys:",
    persisted_fuel
    .select("fuel_key")
    .distinct()
    .count()
)

print(
    "Persisted Null Airline Keys:",
    persisted_fuel
    .filter(col("airline_key").isNull())
    .count()
)

print(
    "Persisted Distinct Airline Keys:",
    persisted_fuel
    .select("airline_key")
    .distinct()
    .count()
)

StatementMeta(, 66d20bab-0403-4f11-afbd-0bfce45d5dc3, 14, Finished, Available, Finished, False)

fact_fuel_consumption persisted successfully.
Persisted Fuel Rows: 53
Persisted Distinct Fuel Keys: 53
Persisted Null Airline Keys: 0
Persisted Distinct Airline Keys: 53
